In [1]:
import os
from datetime import datetime

# Delta Lake
from delta.tables import DeltaTable

# PySpark Core & Types
from pyspark.sql import Row, SparkSession
from pyspark.sql.types import StructType, StructField, DateType, StringType, MapType, DoubleType, LongType
from pyspark.sql.window import Window

# PySpark Functions
from pyspark.sql.functions import (
    col, 
    date_format, 
    explode, 
    explode_outer, 
    input_file_name, 
    lit, 
    rank, 
    regexp_extract, 
    to_timestamp
)


StatementMeta(, c2430cfe-76cf-4a4f-8eb2-da62da0b3c33, 5, Finished, Available, Finished, False)

In [2]:
# Ensure we have the Spark context
spark = SparkSession.builder.getOrCreate()
today_str = datetime.now().strftime("%Y-%m-%d")

StatementMeta(, c2430cfe-76cf-4a4f-8eb2-da62da0b3c33, 6, Finished, Available, Finished, False)

# Stg_most_active_stocks

In [3]:
file_path = f"Files/bronze/{today_str}/most_active_stocks.json"

table_name = "stg_most_active_stocks"

print(f"Starting pipeline for {today_str}...")

# --- THE SAFEGUARD: Check if today's data is already loaded ---
already_loaded = False
if spark.catalog.tableExists(table_name):
    # Quick Spark SQL check to see if today's date exists in the table
    if spark.sql(f"SELECT 1 FROM {table_name} WHERE date = '{today_str}' LIMIT 1").count() > 0:
        already_loaded = True

if already_loaded:
    # 🚨 ONLY PRINTING: This allows the notebook to continue to the next cell!
    print(f"⏭️ Data for {today_str} already exists in '{table_name}'. Skipping processing.")
    
else:
    print(f"Reading from: {file_path}")

    # 1. Read the JSON file
    df_spark = spark.read.option("multiline", "true").json(file_path)

    # 2. Add the date column
    df_spark = df_spark.withColumn("date", lit(today_str))

    # 3. Cast volume to numeric for correct mathematical ordering
    df_spark = df_spark.withColumn("volume", col("volume").cast("long"))

    # 4. Partition by date so ranking resets per day
    window_spec = Window.partitionBy("date").orderBy(col("volume").desc())

    # 5. Apply the window function to generate the rank
    df_spark = df_spark.withColumn("rank", rank().over(window_spec))

    # 6. Sort the final output for clear viewing
    df_spark = df_spark.orderBy("date", "rank")

    # 7. Write to the Lakehouse as a Delta table
    (df_spark.write
        .format("delta")
        .mode("append") 
        .option("mergeSchema", "true") 
        .saveAsTable(table_name))

    print(f"✅ Successfully saved structured data to Delta table: {table_name}")

    # Display the final structured output
    display(df_spark)

StatementMeta(, c2430cfe-76cf-4a4f-8eb2-da62da0b3c33, 7, Finished, Available, Finished, False)

Starting pipeline for 2026-06-19...
Reading from: Files/bronze/2026-06-19/most_active_stocks.json


AnalysisException: [PATH_NOT_FOUND] Path does not exist: abfss://63841cff-dce9-47d4-9cfc-64b7fa24b8a6@onelake.dfs.fabric.microsoft.com/d84d9b55-1b55-4759-b313-1d18335f8517/Files/bronze/2026-06-19/most_active_stocks.json.

# Stg_news

In [ ]:
table_name = "stg_news"

folder_path = f"Files/bronze/{today_str}/news/*_news.json"

print(f"Starting Batch News Processing for {today_str}...")

try:
    # 1. Read ALL news JSON files at once
    df_raw = spark.read.option("multiline", "true").json(folder_path)
    
    # 2. Dynamically extract the Symbol from the file name
    df_file = df_raw.withColumn("file_path", input_file_name())
    df_symbol = df_file.withColumn("symbol", regexp_extract(col("file_path"), r"([^/]+)_stocks_news\.json$", 1))

    # 3. Flatten the 'feed' array
    df_feed = df_symbol.select("symbol", explode_outer("feed").alias("article"))

    # 4. Extract the article-level columns
    df_final = df_feed.select(
        col("symbol"),
        col("article.url").alias("url"),
        col("article.title").alias("title"),
        col("article.summary").alias("summary"),
        date_format(
            to_timestamp(col("article.time_published"), "yyyyMMdd'T'HHmmss"), 
            "dd/MM/yyyy"
        ).alias("time_published"),
        col("article.overall_sentiment_label").alias("overall_sentiment_label"),
        col("article.overall_sentiment_score").alias("overall_sentiment_score")
    )

    # 5. Data Quality: Drop nulls and add extract_date
    df_final = df_final.filter(col("url").isNotNull())
    df_final = df_final.withColumn("extract_date", lit(today_str))
    df_final = df_final.dropDuplicates(["symbol", "url"])

    # --- THE TIME & RESOURCE SAVER LOGIC ---
    if spark.catalog.tableExists(table_name):
        
        # A. Query the table to see which symbols we ALREADY loaded today
        # We use DISTINCT so it returns a tiny, lightning-fast list
        loaded_df = spark.sql(f"SELECT DISTINCT symbol FROM {table_name} WHERE extract_date = '{today_str}'")
        loaded_symbols = [row['symbol'] for row in loaded_df.collect()]
        
        # B. If we found already-loaded symbols, filter them OUT of our incoming DataFrame
        if loaded_symbols:
            print(f"Symbols already processed for {today_str}: {loaded_symbols}")
            df_final = df_final.filter(~col("symbol").isin(loaded_symbols))
            
        # C. Check if we have any data left to write
        if df_final.isEmpty():
            print(f"⏭️ All data for these symbols is already in the table for {today_str}. Skipping write to save compute!")
        else:
            new_record_count = df_final.count()
            print(f"Executing fast APPEND for {new_record_count} new articles...")
            
            # D. Pure APPEND (Massively faster than a MERGE)
            (df_final.write
                .format("delta")
                .mode("append")
                .saveAsTable(table_name))
            
            print(f"✅ Successfully appended new articles to {table_name}")

    else:
        # First-time run
        print(f"Table does not exist. Creating {table_name}...")
        (df_final.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name))
        print(f"✅ Successfully created {table_name}")

    # Display a sample of the structured table
    display(spark.sql(f"SELECT symbol, time_published, title, extract_date FROM {table_name} LIMIT 5"))

except Exception as e:
    print(f"⏭️ Skipping pipeline. Could not process news files. Error: {e}")

StatementMeta(, c2430cfe-76cf-4a4f-8eb2-da62da0b3c33, -1, Cancelled, , Cancelled, True)

# biz_info_lookup

In [ ]:
table_name = "business_info"

folder_path = f"Files/bronze/{today_str}/business_info/*.json"

print(f"Starting Batch Business Info Merge for {today_str}...")

try:
    # 1. Read ALL JSON files in the folder
    df_raw = spark.read.option("multiline", "true").json(folder_path)
    
    # 2. Extract the actual data from the nested 'rows' array format
    if "rows" in df_raw.columns:
        df_spark = df_raw.select(explode("rows").alias("row")).select("row.*")
    else:
        df_spark = df_raw

    # --- THE DATA QUALITY GATE ---
    # 3. Drop any rows where the Primary Key 'Symbol' is completely null
    if "Symbol" in df_spark.columns:
        df_spark = df_spark.filter(col("Symbol").isNotNull())
    else:
        # If the JSON was so broken it didn't even infer a Symbol column
        df_spark = spark.createDataFrame([], schema=df_spark.schema) 

    # 4. Check if we actually have data to process
    if df_spark.isEmpty():
        print(f"⏭️ No valid stock records found in {folder_path} (files were empty or null). Skipping merge.")
    else:
        # 5. Data is valid! Add the 'extract_date' column
        df_spark = df_spark.withColumn("extract_date", lit(today_str))

        # 6. Perform the conditional Delta Merge
        if spark.catalog.tableExists(table_name):
            delta_table = DeltaTable.forName(spark, table_name)
            
            compare_cols = [c for c in df_spark.columns if c not in ["Symbol", "extract_date"]]
            diff_conditions = [f"(NOT (target.{c} <=> source.{c}))" for c in compare_cols]
            diff_expr = " OR ".join(diff_conditions)
            
            update_condition = f"target.extract_date != '{today_str}' AND ({diff_expr})"
            
            valid_record_count = df_spark.count()
            print(f"Executing Delta Merge on {valid_record_count} valid stocks...")
            
            (delta_table.alias("target").merge(
                df_spark.alias("source"),
                "target.Symbol = source.Symbol" 
            )
            .whenMatchedUpdateAll(condition=update_condition) 
            .whenNotMatchedInsertAll()                        
            .execute())
            
            print(f"✅ Successfully merged batch data into {table_name}")

        else:
            # First-time run
            print(f"Table does not exist. Creating {table_name} with batch data...")
            (df_spark.write
                .format("delta")
                .mode("overwrite")
                .saveAsTable(table_name))
            print(f"✅ Successfully created {table_name}")

        # Display a subset of the table to verify
        display(spark.sql(f"SELECT Symbol, Name, extract_date FROM {table_name}"))

except Exception as e:
    # Safe fallback just in case the folder doesn't exist yet
    print(f"⏭️ Skipping pipeline. Could not process files. Error: {e}")

StatementMeta(, c2430cfe-76cf-4a4f-8eb2-da62da0b3c33, -1, Cancelled, , Cancelled, True)

# stg_price

In [ ]:
table_name = "stg_prices"

folder_path = f"Files/bronze/{today_str}/price/*_price.json"

print(f"Starting Batch Daily Price Processing for {today_str}...")

try:
    # 1. Define the internal schema for the open/close/volume metrics
    daily_metrics_schema = StructType([
        StructField("1. open", StringType(), True),
        StructField("2. high", StringType(), True),
        StructField("3. low", StringType(), True),
        StructField("4. close", StringType(), True),
        StructField("5. volume", StringType(), True)
    ])

    # 2. Define the main schema forcing 'Time Series (Daily)' to be parsed as a Key-Value Map
    file_schema = StructType([
        StructField("Meta Data", StructType([
            StructField("2. Symbol", StringType(), True)
        ]), True),
        StructField("Time Series (Daily)", MapType(StringType(), daily_metrics_schema), True)
    ])

    # 3. Read the JSON using our strict schema
    df_raw = spark.read.option("multiline", "true").schema(file_schema).json(folder_path)

    # 4. Filter out any malformed files that didn't parse correctly
    df_valid = df_raw.filter(col("`Time Series (Daily)`").isNotNull())

    # --- THE MAGIC: Explode the Dictionary into Rows ---
    # Exploding a Map generates two columns: 'key' (the date) and 'value' (the struct of metrics)
    df_exploded = df_valid.select(
        col("`Meta Data`.`2. Symbol`").alias("symbol"),
        explode(col("`Time Series (Daily)`")).alias("trade_date", "metrics")
    )

    # 5. Flatten, format, and strictly cast our data types
    df_final = df_exploded.select(
        col("symbol"),
        col("trade_date").cast(DateType()),
        col("metrics.`1. open`").cast(DoubleType()).alias("open_price"),
        col("metrics.`2. high`").cast(DoubleType()).alias("high_price"),
        col("metrics.`3. low`").cast(DoubleType()).alias("low_price"),
        col("metrics.`4. close`").cast(DoubleType()).alias("close_price"),
        col("metrics.`5. volume`").cast(LongType()).alias("volume")
    ).withColumn("extract_date", lit(today_str))

    # 6. The Upsert Merge
    # Alpha Vantage 'compact' returns the last 100 days. A MERGE ensures we only insert the newest days 
    # and safely overwrite existing history in case prices were retroactively adjusted (like stock splits!).
    if spark.catalog.tableExists(table_name):
        delta_table = DeltaTable.forName(spark, table_name)
        
        print("Executing Delta Merge to upsert latest price histories...")
        
        (delta_table.alias("target").merge(
            df_final.alias("source"),
            "target.symbol = source.symbol AND target.trade_date = source.trade_date" 
        )
        .whenMatchedUpdateAll() 
        .whenNotMatchedInsertAll()              
        .execute())
        
        print(f"✅ Successfully upserted historical prices into {table_name}")

    else:
        # First-time run
        print(f"Table does not exist. Creating {table_name} with initial history...")
        (df_final.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table_name))
        print(f"✅ Successfully created {table_name}")

    # Display a sample of the structured table (ordered to show the newest dates first)
    display(df_final.orderBy(col("trade_date").desc()).limit(10))

except Exception as e:
    print(f"⏭️ Skipping pipeline. Could not process price files. Error: {e}")

StatementMeta(, c2430cfe-76cf-4a4f-8eb2-da62da0b3c33, -1, Cancelled, , Cancelled, True)